# Manual cleaning log

This code creates a manual cleaning log by dirctly comparing the original file with the manually cleaned one. 

In [ ]:
import json
import csv
from pathlib import Path
from collections import defaultdict, Counter

# ==================================================
# BASIS-PFADE (AN DEINE STRUKTUR ANGEPASST)
# ==================================================
EXTRACTED_DIR = Path("./../output")
CLEANED_DIR = Path("./../data/datasets/manually_cleaned")
OUTPUT_DIR = Path("./../data/datasets/cleaning_audit")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ==================================================
# AGENCY-NAMEN NORMALISIEREN
# ==================================================
AGENCY_MAP = {
    # Australia
    "TGA": "AUSTRALIA",
    "Australia": "AUSTRALIA",
    "AUSTRALIA": "AUSTRALIA",

    # Japan
    "PMDA": "JAPAN",
    "Japan": "JAPAN",
    "JAPAN": "JAPAN",

    # EMA
    "EMA": "EMA",

    # Swissmedic
    "SwissMedic": "SWISSMEDIC",
    "SWISSMEDIC": "SWISSMEDIC",
}

# ==================================================
# HILFSFUNKTIONEN
# ==================================================
def normalize_display(value):
    """
    Anzeige-nahe Normalisierung:
    - KEIN semantischer Vergleich
    - exakt das, was im Feld steht (getrimmt)
    """
    if value is None:
        return "MISSING"
    if isinstance(value, list):
        return "; ".join(map(str, value))
    return str(value).strip()

def extract_agency_from_filename(filename: str) -> str:
    parts = filename.split("_")
    if len(parts) < 2:
        return None

    raw_agency = parts[1]
    return AGENCY_MAP.get(raw_agency)

# ==================================================
# EMA-SPEZIALFALL:
# Alle EMA-Extraktionen zusammenführen
# ==================================================
def load_all_extracted_for_agency(extracted_dir: Path, agency: str) -> dict:
    """
    Führt alle extracted_llm JSONs einer Agency zusammen.
    Ignoriert Metadaten und sammelt nur Dokument-Dicts.
    """
    merged = {}

    for f in extracted_dir.glob(f"*_{agency}_*.json"):
        with open(f, "r", encoding="utf-8") as fh:
            data = json.load(fh)

        # Fall 1: Dict (normal)
        if isinstance(data, dict):
            for k, v in data.items():
                if isinstance(v, dict):
                    merged[k] = v

        # Fall 2: Liste (Fallback)
        elif isinstance(data, list):
            for i, rec in enumerate(data):
                if not isinstance(rec, dict):
                    continue
                key = (
                    rec.get("Document_name")
                    or rec.get("document_name")
                    or f"{agency}_{f.stem}_{i}"
                )
                merged[key] = rec

    return merged

# ==================================================
# CORE AUDIT-LOGIK (ROBUST)
# ==================================================
def build_audit(original: dict, cleaned: dict):
    """
    audit[field][cleaned_value] = Counter({original_value: count})

    - nur echte Datensätze (Dicts)
    - nur wenn original != cleaned (raw string comparison)
    """

    audit = defaultdict(lambda: defaultdict(Counter))

    common_keys = set(original.keys()) & set(cleaned.keys())

    for doc_key in common_keys:
        orig_rec = original.get(doc_key)
        cln_rec  = cleaned.get(doc_key)

        # 🔒 ROBUSTHEIT: nur echte Records vergleichen
        if not isinstance(orig_rec, dict):
            continue
        if not isinstance(cln_rec, dict):
            continue

        for field, orig_val in orig_rec.items():
            cln_val = cln_rec.get(field)

            orig_disp = normalize_display(orig_val)
            cln_disp  = normalize_display(cln_val)

            # nur tatsächlich manuell geänderte Werte
            if orig_disp == cln_disp:
                continue

            audit[field][cln_disp][orig_disp] += 1

    return audit

# ==================================================
# CSV: EINE ZEILE PRO CLEANED VALUE
# ==================================================
def write_audit_csv(audit: dict, out_path: Path):
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "field",
            "cleaned_value",
            "total_count",
            "original_variants"
        ])

        for field in sorted(audit.keys()):
            for cleaned_value, counter in audit[field].items():
                total = sum(counter.values())
                variants = "; ".join(
                    f"{orig} ({cnt})" for orig, cnt in counter.most_common()
                )

                writer.writerow([
                    field,
                    cleaned_value,
                    total,
                    variants
                ])

# ==================================================
# MAIN: ALLE AGENCIES
# ==================================================
for extracted_file in EXTRACTED_DIR.glob("*.json"):

    agency = extract_agency_from_filename(extracted_file.name)
    if agency is None:
        print(f"⚠️  Agency nicht erkannt: {extracted_file.name}")
        continue

    cleaned_file = CLEANED_DIR / f"{agency}_manually_cleaned.json"
    if not cleaned_file.exists():
        print(f"⚠️  Kein cleaned file für {agency} – übersprungen")
        continue

    print(f"▶️  Verarbeite {agency}")

    # ------------------------------
    # ORIGINALDATEN LADEN
    # ------------------------------
    if agency == "EMA":
        # 🔑 WICHTIGER FIX: alle EMA-Extraktionen zusammenführen
        original_data = load_all_extracted_for_agency(EXTRACTED_DIR, "EMA")
    else:
        with open(extracted_file, "r", encoding="utf-8") as f:
            raw = json.load(f)

        # nur echte Dokument-Dicts behalten
        original_data = {
            k: v for k, v in raw.items()
            if isinstance(v, dict)
        }

    # ------------------------------
    # CLEANED DATEN LADEN
    # ------------------------------
    with open(cleaned_file, "r", encoding="utf-8") as f:
        raw_clean = json.load(f)

    cleaned_data = {
        k: v for k, v in raw_clean.items()
        if isinstance(v, dict)
    }

    # ------------------------------
    # AUDIT
    # ------------------------------
    audit = build_audit(original_data, cleaned_data)

    output_csv = OUTPUT_DIR / f"{agency}_cleaning_audit.csv"
    write_audit_csv(audit, output_csv)

    print(f"   ✅ gespeichert: {output_csv}")

print("\n🎉 Alle verfügbaren Agencies erfolgreich verarbeitet.")


▶️  Verarbeite SwissMedic
   ✅ gespeichert: ../data/datasets/cleaning_audit/SwissMedic_cleaning_audit.csv
▶️  Verarbeite EMA
   ✅ gespeichert: ../data/datasets/cleaning_audit/EMA_cleaning_audit.csv
⚠️  Kein cleaned file für TGA – übersprungen
⚠️  Kein cleaned file für PMDA – übersprungen

🎉 Alle verfügbaren Agencies erfolgreich verarbeitet.
